In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project data folder
DATA_DIR = Path.cwd()

# Expected project files
data_files = [
    "demographics_clean.csv",
    "socioeconomic_education_clean.csv",
    "healthcare_workforce_clean.csv",
    "healthcare_capacity_clean.csv",
    "healthcare_coverage_clean.csv",
    "health_financing_clean.csv",
    "financial_access_equity_clean.csv",
]

support_files = [
    "coverage_summary.csv",
    "unmatched_codes_review.csv",
]

print("=" * 65)
print("GLOBAL HEALTHCARE PROJECT — DATA AUDIT")
print("=" * 65)

print(f"\nWorking directory:\n{DATA_DIR}")

print("\n--- Main Datasets ---")

missing_files = []

for filename in data_files:
    file_path = DATA_DIR / filename

    if file_path.exists():
        size_kb = file_path.stat().st_size / 1024
        print(f"[FOUND]  {filename} ({size_kb:.2f} KB)")
    else:
        print(f"[MISSING] {filename}")
        missing_files.append(filename)

print("\n--- Supporting Files ---")

for filename in support_files:
    file_path = DATA_DIR / filename

    if file_path.exists():
        size_kb = file_path.stat().st_size / 1024
        print(f"[FOUND]  {filename} ({size_kb:.2f} KB)")
    else:
        print(f"[MISSING] {filename}")
        missing_files.append(filename)

print("\n" + "=" * 65)
print(f"Expected main datasets: {len(data_files)}")
print(f"Total missing files: {len(missing_files)}")

if not missing_files:
    print("STATUS: All expected files are available.")
else:
    print("STATUS: Some files need to be located before proceeding.")

print("=" * 65)

GLOBAL HEALTHCARE PROJECT — DATA AUDIT

Working directory:
c:\Users\talal\OneDrive\Desktop\Data after work

--- Main Datasets ---
[FOUND]  demographics_clean.csv (992.32 KB)
[FOUND]  socioeconomic_education_clean.csv (421.68 KB)
[FOUND]  healthcare_workforce_clean.csv (170.00 KB)
[FOUND]  healthcare_capacity_clean.csv (77.31 KB)
[FOUND]  healthcare_coverage_clean.csv (108.77 KB)
[FOUND]  health_financing_clean.csv (313.27 KB)
[FOUND]  financial_access_equity_clean.csv (72.19 KB)

--- Supporting Files ---
[FOUND]  coverage_summary.csv (3.40 KB)
[FOUND]  unmatched_codes_review.csv (0.37 KB)

Expected main datasets: 7
Total missing files: 0
STATUS: All expected files are available.


In [2]:
# Cell 2: Load and inspect all main datasets

datasets = {}
load_errors = {}

for filename in data_files:
    file_path = DATA_DIR / filename

    try:
        df = pd.read_csv(file_path, low_memory=False)
        datasets[filename] = df
        print(f"[LOADED] {filename} | Shape: {df.shape}")

    except Exception as e:
        load_errors[filename] = str(e)
        print(f"[ERROR] {filename} | {e}")


# Build a structural and key-quality summary
audit_results = []

for filename, df in datasets.items():

    has_iso3 = "ISO3" in df.columns
    has_year = "Year" in df.columns
    has_country = "Country" in df.columns

    duplicate_keys = (
        int(df.duplicated(subset=["ISO3", "Year"]).sum())
        if has_iso3 and has_year
        else None
    )

    missing_key_values = (
        int(df[["ISO3", "Year"]].isna().sum().sum())
        if has_iso3 and has_year
        else None
    )

    audit_results.append({
        "Dataset": filename,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Has_ISO3": has_iso3,
        "Has_Country": has_country,
        "Has_Year": has_year,
        "Duplicate_ISO3_Year": duplicate_keys,
        "Missing_Key_Values": missing_key_values
    })

audit_summary = pd.DataFrame(audit_results)

print("\n" + "=" * 90)
print("DATASET STRUCTURE & KEY AUDIT")
print("=" * 90)

display(audit_summary)


# Print actual column names for every dataset
print("\n" + "=" * 90)
print("ACTUAL COLUMN NAMES")
print("=" * 90)

for filename, df in datasets.items():
    print(f"\n{filename}")
    print(list(df.columns))


# Report loading errors, if any
if load_errors:
    print("\nFILES THAT COULD NOT BE LOADED:")
    for filename, error in load_errors.items():
        print(f"- {filename}: {error}")
else:
    print("\nAll seven main datasets loaded successfully.")

[LOADED] demographics_clean.csv | Shape: (12804, 7)
[LOADED] socioeconomic_education_clean.csv | Shape: (8634, 6)
[LOADED] healthcare_workforce_clean.csv | Shape: (4624, 6)
[LOADED] healthcare_capacity_clean.csv | Shape: (2958, 4)
[LOADED] healthcare_coverage_clean.csv | Shape: (4680, 4)
[LOADED] health_financing_clean.csv | Shape: (4612, 6)
[LOADED] financial_access_equity_clean.csv | Shape: (1088, 12)

DATASET STRUCTURE & KEY AUDIT


,Dataset,Rows,Columns,Has_ISO3,Has_Country,Has_Year,Duplicate_ISO3_Year,Missing_Key_Values
0,demographics_clean.csv,12804,7,True,True,True,0.0,0.0
1,socioeconomic_education_clean.csv,8634,6,True,True,True,0.0,0.0
2,healthcare_workforce_clean.csv,4624,6,True,True,True,0.0,0.0
3,healthcare_capacity_clean.csv,2958,4,True,False,True,0.0,0.0
4,healthcare_coverage_clean.csv,4680,4,True,True,True,0.0,0.0
5,health_financing_clean.csv,4612,6,True,True,True,0.0,0.0
6,financial_access_equity_clean.csv,1088,12,True,True,False,NaN,NaN



ACTUAL COLUMN NAMES

demographics_clean.csv
['ISO3', 'Country', 'Year', 'Population', 'Age_0_14_Pct', 'Age_65_Plus_Pct', 'Urban_Population_Pct']

socioeconomic_education_clean.csv
['ISO3', 'Country', 'Year', 'GDP_Per_Capita_PPP_2021', 'Secondary_Enrollment_Gross_Pct', 'Adult_Literacy_Rate']

healthcare_workforce_clean.csv
['ISO3', 'Country', 'Year', 'Doctors_Total', 'Doctors_Per_10k', 'Nurses_Midwives_Per_10k']

healthcare_capacity_clean.csv
['ISO3', 'Country_Source', 'Year', 'Hospital_Beds_Per_10k']

healthcare_coverage_clean.csv
['ISO3', 'Country', 'Year', 'UHC_Service_Coverage_Index']

health_financing_clean.csv
['ISO3', 'Country', 'Year', 'Health_Exp_Per_Capita_PPP', 'Health_Exp_Pct_GDP', 'OOP_Pct_Current_Health_Exp']

financial_access_equity_clean.csv
['ISO3', 'Country', 'Survey_Year', 'Financial_Hardship_Total_Pct', 'Large_Spending_Pct', 'Impoverishing_Spending_Pct', 'Q1_Poorest_Pct', 'Q2_Pct', 'Q3_Pct', 'Q4_Pct', 'Q5_Richest_Pct', 'Equity_Gap_Pct_Points']

All seven main datase

In [3]:
# Cell 3: Detailed Data Quality Audit

quality_results = []
missing_results = []

for filename, df in datasets.items():

    # Detect the appropriate year column
    if "Year" in df.columns:
        year_col = "Year"
    elif "Survey_Year" in df.columns:
        year_col = "Survey_Year"
    else:
        year_col = None

    # Identify the country-name column, if available
    country_col = next(
        (
            col for col in ["Country", "Country_Source"]
            if col in df.columns
        ),
        None
    )

    # Country and year coverage
    unique_countries = (
        df["ISO3"].nunique()
        if "ISO3" in df.columns
        else np.nan
    )

    year_values = (
        pd.to_numeric(df[year_col], errors="coerce")
        if year_col
        else pd.Series(dtype="float64")
    )

    min_year = year_values.min() if not year_values.empty else np.nan
    max_year = year_values.max() if not year_values.empty else np.nan

    # Check duplicate country-year keys where possible
    if "ISO3" in df.columns and year_col:
        duplicate_keys = int(
            df.duplicated(subset=["ISO3", year_col]).sum()
        )
        missing_keys = int(
            df[["ISO3", year_col]].isna().sum().sum()
        )
    else:
        duplicate_keys = np.nan
        missing_keys = np.nan

    quality_results.append({
        "Dataset": filename,
        "Rows": len(df),
        "Unique_ISO3": unique_countries,
        "Year_Column": year_col,
        "Min_Year": min_year,
        "Max_Year": max_year,
        "Duplicate_Country_Year": duplicate_keys,
        "Missing_Key_Values": missing_keys,
        "Country_Name_Column": country_col
    })

    # Missing values and data types for every column
    for col in df.columns:
        missing_count = int(df[col].isna().sum())

        missing_results.append({
            "Dataset": filename,
            "Column": col,
            "Data_Type": str(df[col].dtype),
            "Missing_Count": missing_count,
            "Missing_Pct": round(
                100 * missing_count / len(df), 2
            ) if len(df) else np.nan
        })


quality_summary = pd.DataFrame(quality_results)
missing_summary = pd.DataFrame(missing_results)

print("=" * 100)
print("DATASET-LEVEL QUALITY SUMMARY")
print("=" * 100)

display(quality_summary)

print("\n" + "=" * 100)
print("MISSING VALUES AND DATA TYPES")
print("=" * 100)

display(
    missing_summary.sort_values(
        ["Dataset", "Missing_Count"],
        ascending=[True, False]
    ).reset_index(drop=True)
)

print("\nAudit completed. No source files were modified.")

DATASET-LEVEL QUALITY SUMMARY


,Dataset,Rows,Unique_ISO3,Year_Column,Min_Year,Max_Year,Duplicate_Country_Year,Missing_Key_Values,Country_Name_Column
0,demographics_clean.csv,12804,194,Year,1960,2025,0,0,Country
1,socioeconomic_education_clean.csv,8634,194,Year,1970,2025,0,0,Country
2,healthcare_workforce_clean.csv,4624,194,Year,1971,2024,0,0,Country
3,healthcare_capacity_clean.csv,2958,193,Year,2000,2023,0,0,Country_Source
4,healthcare_coverage_clean.csv,4680,195,Year,2000,2023,0,0,Country
5,health_financing_clean.csv,4612,195,Year,2000,2024,0,0,Country
6,financial_access_equity_clean.csv,1088,168,Survey_Year,1985,2024,0,0,Country



MISSING VALUES AND DATA TYPES


,Dataset,Column,Data_Type,Missing_Count,Missing_Pct
0,demographics_clean.csv,Population,float64,30,0.23
1,demographics_clean.csv,ISO3,object,0,0.00
2,demographics_clean.csv,Country,object,0,0.00
3,demographics_clean.csv,Year,int64,0,0.00
4,demographics_clean.csv,Age_0_14_Pct,float64,0,0.00
5,demographics_clean.csv,Age_65_Plus_Pct,float64,0,0.00
6,demographics_clean.csv,Urban_Population_Pct,float64,0,0.00
7,financial_access_equity_clean.csv,Q1_Poorest_Pct,float64,41,3.77
8,financial_access_equity_clean.csv,Q2_Pct,float64,41,3.77
9,financial_access_equity_clean.csv,Q3_Pct,float64,41,3.77



Audit completed. No source files were modified.


In [4]:
# Cell 4: Domain and Value Range Validation

range_rules = {
    "demographics_clean.csv": {
        "Population": (0, None),
        "Age_0_14_Pct": (0, 100),
        "Age_65_Plus_Pct": (0, 100),
        "Urban_Population_Pct": (0, 100),
    },

    "socioeconomic_education_clean.csv": {
        "GDP_Per_Capita_PPP_2021": (0, None),
        "Secondary_Enrollment_Gross_Pct": (0, None),
        "Adult_Literacy_Rate": (0, 100),
    },

    "healthcare_workforce_clean.csv": {
        "Doctors_Total": (0, None),
        "Doctors_Per_10k": (0, None),
        "Nurses_Midwives_Per_10k": (0, None),
    },

    "healthcare_capacity_clean.csv": {
        "Hospital_Beds_Per_10k": (0, None),
    },

    "healthcare_coverage_clean.csv": {
        "UHC_Service_Coverage_Index": (0, 100),
    },

    "health_financing_clean.csv": {
        "Health_Exp_Per_Capita_PPP": (0, None),
        "Health_Exp_Pct_GDP": (0, 100),
        "OOP_Pct_Current_Health_Exp": (0, 100),
    },

    "financial_access_equity_clean.csv": {
        "Financial_Hardship_Total_Pct": (0, 100),
        "Large_Spending_Pct": (0, 100),
        "Impoverishing_Spending_Pct": (0, 100),
        "Q1_Poorest_Pct": (0, 100),
        "Q2_Pct": (0, 100),
        "Q3_Pct": (0, 100),
        "Q4_Pct": (0, 100),
        "Q5_Richest_Pct": (0, 100),
    },
}

range_results = []

for filename, rules in range_rules.items():

    df = datasets[filename]

    for column, (lower_bound, upper_bound) in rules.items():

        if column not in df.columns:
            range_results.append({
                "Dataset": filename,
                "Column": column,
                "Status": "COLUMN MISSING",
                "Min_Value": np.nan,
                "Max_Value": np.nan,
                "Missing_Values": np.nan,
                "Below_Min": np.nan,
                "Above_Max": np.nan,
                "Non_Numeric_NonMissing": np.nan,
            })
            continue

        original = df[column]
        numeric = pd.to_numeric(original, errors="coerce")

        non_numeric = int(
            (original.notna() & numeric.isna()).sum()
        )

        below_min = (
            int((numeric < lower_bound).sum())
            if lower_bound is not None
            else 0
        )

        above_max = (
            int((numeric > upper_bound).sum())
            if upper_bound is not None
            else 0
        )

        if non_numeric > 0:
            status = "REVIEW: NON-NUMERIC VALUES"
        elif below_min > 0 or above_max > 0:
            status = "REVIEW: OUT-OF-RANGE VALUES"
        else:
            status = "PASS"

        range_results.append({
            "Dataset": filename,
            "Column": column,
            "Status": status,
            "Min_Value": numeric.min(),
            "Max_Value": numeric.max(),
            "Missing_Values": int(original.isna().sum()),
            "Below_Min": below_min,
            "Above_Max": above_max,
            "Non_Numeric_NonMissing": non_numeric,
        })


range_summary = pd.DataFrame(range_results)

print("=" * 100)
print("DOMAIN AND VALUE RANGE VALIDATION")
print("=" * 100)

display(range_summary)

print("\nStatus counts:")
display(range_summary["Status"].value_counts().rename("Count"))

print("\nNo source dataset was modified.")

DOMAIN AND VALUE RANGE VALIDATION


,Dataset,Column,Status,Min_Value,Max_Value,Missing_Values,Below_Min,Above_Max,Non_Numeric_NonMissing
0,demographics_clean.csv,Population,PASS,4607.000000,1.463866e+09,30,0,0,0
1,demographics_clean.csv,Age_0_14_Pct,PASS,10.204467,5.237937e+01,0,0,0,0
2,demographics_clean.csv,Age_65_Plus_Pct,PASS,0.862465,3.731928e+01,0,0,0,0
3,demographics_clean.csv,Urban_Population_Pct,PASS,1.838567,1.000000e+02,0,0,0,0
4,socioeconomic_education_clean.csv,GDP_Per_Capita_PPP_2021,PASS,510.822823,1.490251e+05,2004,0,0,0
5,socioeconomic_education_clean.csv,Secondary_Enrollment_Gross_Pct,PASS,0.000000,1.640798e+02,2611,0,0,0
6,socioeconomic_education_clean.csv,Adult_Literacy_Rate,PASS,8.830000,1.000000e+02,7532,0,0,0
7,healthcare_workforce_clean.csv,Doctors_Total,PASS,3.000000,4.009658e+06,331,0,0,0
8,healthcare_workforce_clean.csv,Doctors_Per_10k,PASS,0.130000,9.543000e+01,344,0,0,0
9,healthcare_workforce_clean.csv,Nurses_Midwives_Per_10k,PASS,0.470000,2.083100e+02,1250,0,0,0



Status counts:


Status
PASS    23
Name: Count, dtype: int64


No source dataset was modified.


In [5]:
# Cell 5: Country Code and Name Consistency Audit

import re

country_mapping_records = []
mapping_quality_results = []

for filename, df in datasets.items():

    # 1. Validate ISO3 format
    iso_codes = df["ISO3"].astype("string")
    normalized_codes = iso_codes.str.strip().str.upper()

    invalid_iso3_mask = (
        normalized_codes.isna()
        | ~normalized_codes.str.fullmatch(r"[A-Z]{3}", na=False)
    )

    invalid_iso3_count = int(invalid_iso3_mask.sum())

    # 2. Identify available country-name column
    country_col = next(
        (
            col for col in ["Country", "Country_Source"]
            if col in df.columns
        ),
        None
    )

    if country_col is not None:

        country_names = df[country_col].astype("string").str.strip()

        missing_country_count = int(
            country_names.isna().sum()
            + country_names.eq("").fillna(False).sum()
        )

        mapping_df = pd.DataFrame({
            "Dataset": filename,
            "ISO3": normalized_codes,
            "Country_Name": country_names
        })

        mapping_df["Normalized_Name"] = (
            mapping_df["Country_Name"]
            .str.casefold()
            .str.strip()
        )

        mapping_df = mapping_df[
            mapping_df["ISO3"].notna()
            & mapping_df["Normalized_Name"].notna()
            & mapping_df["Normalized_Name"].ne("")
        ]

        country_mapping_records.append(mapping_df)

        # Check whether one ISO3 maps to multiple names
        names_per_code = (
            mapping_df.groupby("ISO3")["Normalized_Name"]
            .nunique()
        )

        ambiguous_codes = names_per_code[names_per_code > 1]

        ambiguous_code_count = len(ambiguous_codes)

    else:
        missing_country_count = None
        ambiguous_code_count = None

    mapping_quality_results.append({
        "Dataset": filename,
        "Invalid_ISO3_Rows": invalid_iso3_count,
        "Missing_Country_Names": missing_country_count,
        "ISO3_With_Multiple_Names_Within_Dataset":
            ambiguous_code_count
    })


mapping_quality_summary = pd.DataFrame(mapping_quality_results)

print("=" * 100)
print("COUNTRY MAPPING QUALITY BY DATASET")
print("=" * 100)

display(mapping_quality_summary)


# Compare country names across datasets
if country_mapping_records:

    all_country_names = pd.concat(
        country_mapping_records,
        ignore_index=True
    )

    cross_dataset_mapping = (
        all_country_names
        .groupby("ISO3")
        .agg(
            Number_Of_Name_Variants=("Normalized_Name", "nunique"),
            Names=("Country_Name",
                   lambda x: " | ".join(sorted(set(x.dropna())))),
            Datasets=("Dataset",
                      lambda x: " | ".join(sorted(set(x))))
        )
        .reset_index()
    )

    name_conflicts = cross_dataset_mapping[
        cross_dataset_mapping["Number_Of_Name_Variants"] > 1
    ].copy()

    print("\n" + "=" * 100)
    print("COUNTRY NAMES THAT DIFFER ACROSS DATASETS")
    print("=" * 100)

    if name_conflicts.empty:
        print("No cross-dataset country-name differences were detected.")
    else:
        display(name_conflicts)

else:
    print("No country-name columns were available for comparison.")


print("\nAudit completed. Original datasets remain unchanged.")

COUNTRY MAPPING QUALITY BY DATASET


,Dataset,Invalid_ISO3_Rows,Missing_Country_Names,ISO3_With_Multiple_Names_Within_Dataset
0,demographics_clean.csv,0,0,0
1,socioeconomic_education_clean.csv,0,0,0
2,healthcare_workforce_clean.csv,0,0,10
3,healthcare_capacity_clean.csv,0,0,0
4,healthcare_coverage_clean.csv,0,0,0
5,health_financing_clean.csv,0,0,0
6,financial_access_equity_clean.csv,0,9,0



COUNTRY NAMES THAT DIFFER ACROSS DATASETS


,ISO3,Number_Of_Name_Variants,Names,Datasets
22,BOL,2,"Bolivia (Plurinational State of) | Bolivia, Pl...",demographics_clean.csv | financial_access_equi...
33,CIV,2,Cote d'Ivoire | Côte d'Ivoire,demographics_clean.csv | financial_access_equi...
35,COD,2,"Congo, The Democratic Republic of the | Democr...",demographics_clean.csv | financial_access_equi...
60,FSM,2,"Micronesia (Federated States of) | Micronesia,...",demographics_clean.csv | financial_access_equi...
62,GBR,2,United Kingdom | United Kingdom of Great Brita...,demographics_clean.csv | financial_access_equi...
80,IRN,2,"Iran (Islamic Republic of) | Iran, Islamic Rep...",demographics_clean.csv | financial_access_equi...
94,KOR,2,"Korea, Republic of | Republic of Korea",demographics_clean.csv | financial_access_equi...
109,MDA,2,"Moldova, Republic of | Republic of Moldova",demographics_clean.csv | financial_access_equi...
130,NLD,2,Netherlands | Netherlands (Kingdom of the),demographics_clean.csv | financial_access_equi...
133,NRU,2,Naoero | Nauru,demographics_clean.csv | financial_access_equi...



Audit completed. Original datasets remain unchanged.


In [6]:
# Cell 6: Investigate Multiple Country Names in Healthcare Workforce

workforce = datasets["healthcare_workforce_clean.csv"].copy()

# Count distinct country names associated with each ISO3
name_counts = (
    workforce.groupby("ISO3")["Country"]
    .nunique(dropna=True)
)

# Identify ISO3 codes associated with multiple country names
problem_codes = name_counts[name_counts > 1].index.tolist()

print("=" * 90)
print("COUNTRY NAME VARIANTS WITHIN HEALTHCARE WORKFORCE")
print("=" * 90)

print(f"Number of affected ISO3 codes: {len(problem_codes)}")

# Display each affected code, its names, year range, and record count
for code in sorted(problem_codes):

    subset = workforce.loc[
        workforce["ISO3"] == code,
        ["ISO3", "Country", "Year"]
    ].copy()

    summary = (
        subset.groupby("Country", dropna=False)
        .agg(
            Records=("Year", "size"),
            First_Year=("Year", "min"),
            Last_Year=("Year", "max")
        )
        .reset_index()
    )

    print(f"\nISO3: {code}")
    display(summary)

print("\nInvestigation completed. No source data was modified.")

COUNTRY NAME VARIANTS WITHIN HEALTHCARE WORKFORCE
Number of affected ISO3 codes: 10

ISO3: BOL


,Country,Records,First_Year,Last_Year
0,Bolivia (Plurinational State of),9,2001,2021
1,"Bolivia, Plurinational State of",1,2015,2015



ISO3: CIV


,Country,Records,First_Year,Last_Year
0,Cote d'Ivoire,12,2004,2024
1,Côte d'Ivoire,4,2013,2017



ISO3: COD


...


ISO3: USA


,Country,Records,First_Year,Last_Year
0,United States,3,1990,1992
1,United States of America,31,1985,2022



ISO3: VEN


,Country,Records,First_Year,Last_Year
0,Venezuela (Bolivarian Republic of),2,2001,2017
1,"Venezuela, Bolivarian Republic of",2,1997,2018



Investigation completed. No source data was modified.


In [7]:
# Cell 7: Locate the approved country dictionary

from pathlib import Path

search_roots = [
    DATA_DIR,
    DATA_DIR.parent / "Data_Raw_01",
    DATA_DIR.parent / "ملفات تسكات المشروع",
]

dictionary_candidates = []

for root in search_roots:
    if root.exists():
        for path in root.rglob("country_dictionary_v1.csv"):
            if path not in dictionary_candidates:
                dictionary_candidates.append(path)

print("=" * 80)
print("COUNTRY DICTIONARY SEARCH")
print("=" * 80)

if dictionary_candidates:
    for i, path in enumerate(dictionary_candidates, start=1):
        print(f"{i}. {path}")
else:
    print("Dictionary not found in the searched folders.")

print("=" * 80)

COUNTRY DICTIONARY SEARCH
Dictionary not found in the searched folders.


In [8]:
# Cell 8: Search for the country dictionary in broader locations

from pathlib import Path

search_roots = [
    DATA_DIR.parent,
    Path.home() / "OneDrive"
]

dictionary_candidates = []

for root in search_roots:
    if not root.exists():
        continue

    try:
        for path in root.rglob("country_dictionary_v1.csv"):
            if path not in dictionary_candidates:
                dictionary_candidates.append(path)
    except (PermissionError, OSError):
        continue

print("=" * 80)
print("COUNTRY DICTIONARY SEARCH RESULTS")
print("=" * 80)

if dictionary_candidates:
    for i, path in enumerate(dictionary_candidates, start=1):
        print(f"{i}. {path}")
else:
    print("NOT FOUND: country_dictionary_v1.csv")

print("=" * 80)

COUNTRY DICTIONARY SEARCH RESULTS
1. c:\Users\talal\OneDrive\Desktop\ملفات تاسكات المشروع\Task2 Data Analysis\country_dictionary_v1.csv
2. c:\Users\talal\OneDrive\Desktop\ملفات تاسكات المشروع\task 1\Data_Raw_01\country_dictionary_v1.csv
3. c:\Users\talal\OneDrive\Desktop\ملفات تاسكات المشروع\Healthcare_Project  task3\Healthcare_Project\Data_Raw_01\country_dictionary_v1.csv
4. c:\Users\talal\OneDrive\Desktop\data -world\00_Project_Reference\country_dictionary_v1.csv


In [9]:
# Cell 9: Compare all available country dictionary copies

from pathlib import Path
import hashlib
import pandas as pd

dictionary_paths = [
    Path(
        r"C:\Users\talal\OneDrive\Desktop\ملفات تاسكات المشروع\Task2 Data Analysis\country_dictionary_v1.csv"
    ),
    Path(
        r"C:\Users\talal\OneDrive\Desktop\ملفات تاسكات المشروع\task 1\Data_Raw_01\country_dictionary_v1.csv"
    ),
    Path(
        r"C:\Users\talal\OneDrive\Desktop\ملفات تاسكات المشروع\Healthcare_Project  task3\Healthcare_Project\Data_Raw_01\country_dictionary_v1.csv"
    ),
    Path(
        r"C:\Users\talal\OneDrive\Desktop\data -world\00_Project_Reference\country_dictionary_v1.csv"
    ),
]

dictionary_audit = []

for path in dictionary_paths:

    if not path.exists():
        dictionary_audit.append({
            "Path": str(path),
            "Exists": False,
            "Rows": None,
            "Columns": None,
            "Unique_ISO3": None,
            "Duplicate_ISO3": None,
            "SHA256": None,
        })
        continue

    # Read dictionary without modifying the original file
    dictionary_df = pd.read_csv(path)

    # Calculate content hash to compare exact file copies
    file_hash = hashlib.sha256(path.read_bytes()).hexdigest()

    dictionary_audit.append({
        "Path": str(path),
        "Exists": True,
        "Rows": len(dictionary_df),
        "Columns": len(dictionary_df.columns),
        "Unique_ISO3": (
            dictionary_df["ISO3"].nunique()
            if "ISO3" in dictionary_df.columns
            else None
        ),
        "Duplicate_ISO3": (
            int(dictionary_df["ISO3"].duplicated().sum())
            if "ISO3" in dictionary_df.columns
            else None
        ),
        "SHA256": file_hash,
    })

dictionary_audit_df = pd.DataFrame(dictionary_audit)

print("=" * 100)
print("COUNTRY DICTIONARY COPIES COMPARISON")
print("=" * 100)

display(dictionary_audit_df)

available_hashes = dictionary_audit_df.loc[
    dictionary_audit_df["Exists"] == True, "SHA256"
].dropna().unique()

print("\nNumber of distinct file versions:", len(available_hashes))

if len(available_hashes) == 1:
    print("RESULT: All available copies are byte-for-byte identical.")
else:
    print("RESULT: Copies differ or one or more files are unavailable.")
    print("We need to inspect the differences before selecting the canonical copy.")

print("\nNo source dictionary was modified.")

COUNTRY DICTIONARY COPIES COMPARISON


,Path,Exists,Rows,Columns,Unique_ISO3,Duplicate_ISO3,SHA256
0,C:\Users\talal\OneDrive\Desktop\ملفات تاسكات ا...,True,196,13,196,0,0535ecf2fa481cb8d2d787e93b99f08bf49bb5e2d1b2d5...
1,C:\Users\talal\OneDrive\Desktop\ملفات تاسكات ا...,True,196,13,196,0,0535ecf2fa481cb8d2d787e93b99f08bf49bb5e2d1b2d5...
2,C:\Users\talal\OneDrive\Desktop\ملفات تاسكات ا...,True,196,13,196,0,0535ecf2fa481cb8d2d787e93b99f08bf49bb5e2d1b2d5...
3,C:\Users\talal\OneDrive\Desktop\data -world\00...,True,196,13,196,0,0535ecf2fa481cb8d2d787e93b99f08bf49bb5e2d1b2d5...



Number of distinct file versions: 1
RESULT: All available copies are byte-for-byte identical.

No source dictionary was modified.


In [10]:
# Cell 10: Validate all datasets against the approved country dictionary

from pathlib import Path
import pandas as pd

# Canonical country dictionary path
DICTIONARY_PATH = Path(
    r"C:\Users\talal\OneDrive\Desktop\data -world\00_Project_Reference\country_dictionary_v1.csv"
)

# Load the approved dictionary
country_dictionary = pd.read_csv(
    DICTIONARY_PATH,
    dtype={"ISO3": "string"}
)

# Validate required dictionary columns
required_dictionary_columns = {"ISO3", "Country"}

missing_dictionary_columns = (
    required_dictionary_columns - set(country_dictionary.columns)
)

if missing_dictionary_columns:
    raise ValueError(
        f"Required dictionary columns are missing: "
        f"{sorted(missing_dictionary_columns)}"
    )

# Standardize codes for comparison only; source files remain unchanged
country_dictionary["ISO3"] = (
    country_dictionary["ISO3"]
    .astype("string")
    .str.strip()
    .str.upper()
)

if country_dictionary["ISO3"].isna().any():
    raise ValueError("The country dictionary contains missing ISO3 codes.")

if country_dictionary["ISO3"].duplicated().any():
    raise ValueError("The country dictionary contains duplicate ISO3 codes.")

# Build canonical ISO3-to-country mapping
canonical_names = (
    country_dictionary
    .set_index("ISO3")["Country"]
    .astype("string")
    .str.strip()
)

summary_records = []
country_name_issues = []
unmatched_code_records = []

for filename, df in datasets.items():

    # Normalize codes in memory for comparison only
    codes = (
        df["ISO3"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    dataset_codes = set(codes.dropna().unique())
    dictionary_codes = set(country_dictionary["ISO3"].dropna())

    unmatched_codes = sorted(dataset_codes - dictionary_codes)

    # Determine the available country-name column
    country_col = next(
        (
            col for col in ["Country", "Country_Source"]
            if col in df.columns
        ),
        None
    )

    missing_country_names = 0
    rows_with_name_differences = 0
    codes_with_name_differences = 0

    # Record codes absent from the approved dictionary
    for code in unmatched_codes:
        unmatched_code_records.append({
            "Dataset": filename,
            "ISO3": code
        })

    if country_col is not None:

        observed_names = df[country_col].astype("string").str.strip()

        missing_country_names = int(
            observed_names.isna().sum()
            + observed_names.eq("").fillna(False).sum()
        )

        comparison = pd.DataFrame({
            "ISO3": codes,
            "Observed_Country": observed_names
        })

        comparison["Dictionary_Country"] = (
            comparison["ISO3"].map(canonical_names)
        )

        # Compare names case-insensitively.
        # This intentionally does not remove accents or rewrite names.
        observed_normalized = (
            comparison["Observed_Country"]
            .str.casefold()
            .str.strip()
        )

        dictionary_normalized = (
            comparison["Dictionary_Country"]
            .str.casefold()
            .str.strip()
        )

        has_observed_name = (
            comparison["Observed_Country"].notna()
            & comparison["Observed_Country"].ne("")
        )

        has_dictionary_name = (
            comparison["Dictionary_Country"].notna()
            & comparison["Dictionary_Country"].ne("")
        )

        mismatch_mask = (
            has_observed_name
            & has_dictionary_name
            & observed_normalized.ne(dictionary_normalized)
        )

        rows_with_name_differences = int(mismatch_mask.sum())

        codes_with_name_differences = int(
            comparison.loc[mismatch_mask, "ISO3"].nunique()
        )

        # Preserve distinct observed name variants for review
        mismatch_details = (
            comparison.loc[
                mismatch_mask,
                ["ISO3", "Observed_Country", "Dictionary_Country"]
            ]
            .drop_duplicates()
        )

        for _, row in mismatch_details.iterrows():
            country_name_issues.append({
                "Dataset": filename,
                "ISO3": row["ISO3"],
                "Observed_Country": row["Observed_Country"],
                "Dictionary_Country": row["Dictionary_Country"]
            })

    summary_records.append({
        "Dataset": filename,
        "Rows": len(df),
        "Unique_ISO3": len(dataset_codes),
        "ISO3_Not_In_Dictionary": len(unmatched_codes),
        "Missing_Country_Name_Rows": missing_country_names,
        "Rows_With_Name_Differences": rows_with_name_differences,
        "Codes_With_Name_Differences": codes_with_name_differences
    })


dictionary_audit_summary = pd.DataFrame(summary_records)

country_name_issues_df = pd.DataFrame(
    country_name_issues,
    columns=[
        "Dataset",
        "ISO3",
        "Observed_Country",
        "Dictionary_Country"
    ]
)

unmatched_codes_df = pd.DataFrame(
    unmatched_code_records,
    columns=["Dataset", "ISO3"]
)

print("=" * 100)
print("COUNTRY DICTIONARY VALIDATION SUMMARY")
print("=" * 100)

display(dictionary_audit_summary)

print("\n" + "=" * 100)
print("COUNTRY NAME DIFFERENCES — REVIEW ONLY")
print("=" * 100)

if country_name_issues_df.empty:
    print("No country-name differences detected.")
else:
    display(country_name_issues_df)

print("\n" + "=" * 100)
print("ISO3 CODES NOT FOUND IN THE APPROVED DICTIONARY")
print("=" * 100)

if unmatched_codes_df.empty:
    print("All dataset ISO3 codes exist in the approved dictionary.")
else:
    display(unmatched_codes_df)

print("\nAudit completed. No source datasets or dictionary files were modified.")

COUNTRY DICTIONARY VALIDATION SUMMARY


,Dataset,Rows,Unique_ISO3,ISO3_Not_In_Dictionary,Missing_Country_Name_Rows,Rows_With_Name_Differences,Codes_With_Name_Differences
0,demographics_clean.csv,12804,194,0,0,0,0
1,socioeconomic_education_clean.csv,8634,194,0,0,0,0
2,healthcare_workforce_clean.csv,4624,194,0,0,273,14
3,healthcare_capacity_clean.csv,2958,193,0,0,203,15
4,healthcare_coverage_clean.csv,4680,195,0,0,0,0
5,health_financing_clean.csv,4612,195,0,0,0,0
6,financial_access_equity_clean.csv,1088,168,3,9,0,0



COUNTRY NAME DIFFERENCES — REVIEW ONLY


,Dataset,ISO3,Observed_Country,Dictionary_Country
0,healthcare_workforce_clean.csv,BOL,Bolivia (Plurinational State of),"Bolivia, Plurinational State of"
1,healthcare_workforce_clean.csv,CIV,Cote d'Ivoire,Côte d'Ivoire
2,healthcare_workforce_clean.csv,COD,Democratic Republic of the Congo,"Congo, The Democratic Republic of the"
3,healthcare_workforce_clean.csv,FSM,Micronesia (Federated States of),"Micronesia, Federated States of"
4,healthcare_workforce_clean.csv,GBR,United Kingdom of Great Britain and Northern I...,United Kingdom
5,healthcare_workforce_clean.csv,IRN,Iran (Islamic Republic of),"Iran, Islamic Republic of"
6,healthcare_workforce_clean.csv,KOR,Republic of Korea,"Korea, Republic of"
7,healthcare_workforce_clean.csv,MDA,Republic of Moldova,"Moldova, Republic of"
8,healthcare_workforce_clean.csv,NLD,Netherlands (Kingdom of the),Netherlands
9,healthcare_workforce_clean.csv,NRU,Naoero,Nauru



ISO3 CODES NOT FOUND IN THE APPROVED DICTIONARY


,Dataset,ISO3
0,financial_access_equity_clean.csv,TKL
1,financial_access_equity_clean.csv,WLF
2,financial_access_equity_clean.csv,XKX



Audit completed. No source datasets or dictionary files were modified.


In [11]:
# Cell 11: Investigate unmatched ISO3 codes and missing country names

equity = datasets["financial_access_equity_clean.csv"].copy()

# Normalize ISO3 for checking only
equity["ISO3_Check"] = (
    equity["ISO3"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Get the ISO3 codes from the approved dictionary
dictionary_codes = set(
    country_dictionary["ISO3"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Identify records with codes not present in the dictionary
unmatched_mask = ~equity["ISO3_Check"].isin(dictionary_codes)

unmatched_records = equity.loc[unmatched_mask].copy()

print("=" * 90)
print("UNMATCHED ISO3 CODE SUMMARY")
print("=" * 90)

if unmatched_records.empty:
    print("No unmatched codes found.")
else:
    unmatched_summary = (
        unmatched_records
        .groupby("ISO3_Check")
        .agg(
            Record_Count=("Survey_Year", "size"),
            First_Survey_Year=("Survey_Year", "min"),
            Last_Survey_Year=("Survey_Year", "max"),
            Missing_Country_Name=("Country", lambda x: x.isna().sum()),
            Records_With_Any_Missing_Indicator=(
                "Financial_Hardship_Total_Pct",
                lambda x: x.isna().sum()
            )
        )
        .reset_index()
        .rename(columns={"ISO3_Check": "ISO3"})
    )

    display(unmatched_summary)

    print("\nDetailed unmatched records:")

    display(
        unmatched_records[
            [
                "ISO3",
                "Country",
                "Survey_Year",
                "Financial_Hardship_Total_Pct",
                "Large_Spending_Pct",
                "Impoverishing_Spending_Pct",
                "Equity_Gap_Pct_Points"
            ]
        ].sort_values(["ISO3", "Survey_Year"])
    )


# Investigate rows where Country is missing
print("\n" + "=" * 90)
print("RECORDS WITH MISSING COUNTRY NAMES")
print("=" * 90)

missing_country_mask = (
    equity["Country"].isna()
    | equity["Country"].astype("string").str.strip().eq("").fillna(False)
)

missing_country_records = equity.loc[missing_country_mask].copy()

print(f"Number of records with missing country names: {len(missing_country_records)}")

if missing_country_records.empty:
    print("No missing country names found.")
else:
    display(
        missing_country_records[
            [
                "ISO3",
                "Survey_Year",
                "Financial_Hardship_Total_Pct",
                "Large_Spending_Pct",
                "Impoverishing_Spending_Pct",
                "Equity_Gap_Pct_Points"
            ]
        ].sort_values(["ISO3", "Survey_Year"])
    )

print("\nInvestigation completed. No source data was modified.")

UNMATCHED ISO3 CODE SUMMARY


,ISO3,Record_Count,First_Survey_Year,Last_Survey_Year,Missing_Country_Name,Records_With_Any_Missing_Indicator
0,TKL,1,2015,2015,1,0
1,WLF,1,2005,2005,1,0
2,XKX,7,2003,2016,7,0



Detailed unmatched records:


,ISO3,Country,Survey_Year,Financial_Hardship_Total_Pct,Large_Spending_Pct,Impoverishing_Spending_Pct,Equity_Gap_Pct_Points
959,TKL,NaN,2015,1.28,0.00,1.28,6.08
1065,WLF,NaN,2005,0.12,0.00,0.12,0.58
1067,XKX,NaN,2003,14.05,2.78,11.27,37.37
1068,XKX,NaN,2005,17.13,3.79,13.34,44.38
1069,XKX,NaN,2006,15.55,3.17,12.38,44.78
1070,XKX,NaN,2009,16.17,3.95,12.21,44.89
1071,XKX,NaN,2010,17.36,3.42,13.94,49.29
1072,XKX,NaN,2011,13.63,2.87,10.76,45.50
1073,XKX,NaN,2016,13.92,3.99,9.93,50.10



RECORDS WITH MISSING COUNTRY NAMES
Number of records with missing country names: 9


,ISO3,Survey_Year,Financial_Hardship_Total_Pct,Large_Spending_Pct,Impoverishing_Spending_Pct,Equity_Gap_Pct_Points
959,TKL,2015,1.28,0.00,1.28,6.08
1065,WLF,2005,0.12,0.00,0.12,0.58
1067,XKX,2003,14.05,2.78,11.27,37.37
1068,XKX,2005,17.13,3.79,13.34,44.38
1069,XKX,2006,15.55,3.17,12.38,44.78
1070,XKX,2009,16.17,3.95,12.21,44.89
1071,XKX,2010,17.36,3.42,13.94,49.29
1072,XKX,2011,13.63,2.87,10.76,45.50
1073,XKX,2016,13.92,3.99,9.93,50.10



Investigation completed. No source data was modified.


In [12]:
# Cell 12: Document unmatched financial equity records

equity = datasets["financial_access_equity_clean.csv"].copy()

# Normalize ISO3 codes for comparison only
equity["ISO3_Check"] = (
    equity["ISO3"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Use the approved canonical country dictionary
dictionary_codes = set(
    country_dictionary["ISO3"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Identify records whose ISO3 code is absent from the dictionary
unmatched_mask = ~equity["ISO3_Check"].isin(dictionary_codes)

# Create a separate review copy; do not modify the source dataset
equity_unmatched_review = equity.loc[unmatched_mask].copy()

equity_unmatched_review = equity_unmatched_review.drop(
    columns=["ISO3_Check"]
)

equity_unmatched_review["Review_Reason"] = (
    "ISO3 not found in the approved country dictionary. "
    "Excluded from the integrated analytical dataset unless "
    "the project lead approves a change to the country universe."
)

# Save the review report without overwriting any existing file
review_path = DATA_DIR / "financial_equity_unmatched_review.csv"

if review_path.exists():
    raise FileExistsError(
        f"{review_path.name} already exists. "
        "Review the existing file before creating a new version."
    )

equity_unmatched_review.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 80)
print("FINANCIAL EQUITY — UNMATCHED RECORDS REVIEW")
print("=" * 80)

print(f"Unmatched records: {len(equity_unmatched_review)}")
print(
    "Unique unmatched ISO3 codes:",
    equity_unmatched_review["ISO3"].nunique()
)

print("\nRecords by ISO3:")
display(
    equity_unmatched_review
    .groupby("ISO3")
    .size()
    .rename("Record_Count")
    .reset_index()
)

print("\nReview file saved to:")
print(review_path)

print("\nOriginal source dataset was not modified.")

FINANCIAL EQUITY — UNMATCHED RECORDS REVIEW
Unmatched records: 9
Unique unmatched ISO3 codes: 3

Records by ISO3:


,ISO3,Record_Count
0,TKL,1
1,WLF,1
2,XKX,7



Review file saved to:
c:\Users\talal\OneDrive\Desktop\Data after work\financial_equity_unmatched_review.csv

Original source dataset was not modified.


In [13]:
# Cell 13: Standardize datasets for integration
# Original source files remain unchanged.

standardized_datasets = {}
standardization_results = []

# Build the canonical ISO3-to-Country mapping
canonical_names = (
    country_dictionary
    .assign(
        ISO3=lambda x: (
            x["ISO3"]
            .astype("string")
            .str.strip()
            .str.upper()
        )
    )
    .set_index("ISO3")["Country"]
    .astype("string")
    .str.strip()
)

for filename, source_df in datasets.items():

    # Work on a copy; never modify the original DataFrame
    df = source_df.copy()

    original_rows = len(df)

    # Standardize column names where required
    rename_map = {}

    if "Country_Source" in df.columns:
        rename_map["Country_Source"] = "Country"

    if "Survey_Year" in df.columns:
        rename_map["Survey_Year"] = "Year"

    df = df.rename(columns=rename_map)

    # Normalize ISO3 formatting for comparison and integration
    df["ISO3"] = (
        df["ISO3"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    # Validate and standardize Year
    year_numeric = pd.to_numeric(df["Year"], errors="coerce")

    invalid_year_mask = (
        year_numeric.isna()
        | (year_numeric % 1 != 0)
    )

    invalid_year_count = int(invalid_year_mask.sum())

    if invalid_year_count > 0:
        raise ValueError(
            f"{filename} contains {invalid_year_count} invalid Year values."
        )

    df["Year"] = year_numeric.astype("int64")

    # Identify records outside the approved country dictionary
    known_codes = df["ISO3"].isin(canonical_names.index)

    excluded_rows = int((~known_codes).sum())

    # Keep only records with ISO3 codes in the approved dictionary
    # This filtering affects the integration copy only.
    df = df.loc[known_codes].copy()

    # Standardize all country names to the approved dictionary
    df["Country"] = df["ISO3"].map(canonical_names)

    if df["Country"].isna().any():
        raise ValueError(
            f"{filename} has ISO3 codes without canonical country names."
        )

    # Validate the country-year key
    duplicate_count = int(
        df.duplicated(subset=["ISO3", "Year"]).sum()
    )

    missing_key_count = int(
        df[["ISO3", "Year"]].isna().sum().sum()
    )

    # Store prepared copy in memory
    standardized_datasets[filename] = df

    standardization_results.append({
        "Dataset": filename,
        "Original_Rows": original_rows,
        "Rows_Excluded_Not_In_Dictionary": excluded_rows,
        "Rows_After_Standardization": len(df),
        "Unique_ISO3": df["ISO3"].nunique(),
        "Min_Year": df["Year"].min(),
        "Max_Year": df["Year"].max(),
        "Duplicate_ISO3_Year": duplicate_count,
        "Missing_Key_Values": missing_key_count,
        "Country_Column": "Country",
        "Year_Column": "Year"
    })


standardization_summary = pd.DataFrame(standardization_results)

print("=" * 110)
print("STANDARDIZATION SUMMARY")
print("=" * 110)

display(standardization_summary)

# Stop before integration if any key-quality issue is detected
if (
    standardization_summary["Duplicate_ISO3_Year"].gt(0).any()
    or standardization_summary["Missing_Key_Values"].gt(0).any()
):
    print("\nWARNING: Some datasets require further review before integration.")
else:
    print("\nPASS: All prepared datasets have unique ISO3 + Year keys.")
    print("PASS: No missing values were found in the integration keys.")
    print("PASS: Country names were standardized using the approved dictionary.")

print("\nOriginal datasets were not modified.")
print("No integrated dataset has been created yet.")

STANDARDIZATION SUMMARY


,Dataset,Original_Rows,Rows_Excluded_Not_In_Dictionary,Rows_After_Standardization,Unique_ISO3,Min_Year,Max_Year,Duplicate_ISO3_Year,Missing_Key_Values,Country_Column,Year_Column
0,demographics_clean.csv,12804,0,12804,194,1960,2025,0,0,Country,Year
1,socioeconomic_education_clean.csv,8634,0,8634,194,1970,2025,0,0,Country,Year
2,healthcare_workforce_clean.csv,4624,0,4624,194,1971,2024,0,0,Country,Year
3,healthcare_capacity_clean.csv,2958,0,2958,193,2000,2023,0,0,Country,Year
4,healthcare_coverage_clean.csv,4680,0,4680,195,2000,2023,0,0,Country,Year
5,health_financing_clean.csv,4612,0,4612,195,2000,2024,0,0,Country,Year
6,financial_access_equity_clean.csv,1088,9,1079,165,1985,2024,0,0,Country,Year



PASS: All prepared datasets have unique ISO3 + Year keys.
PASS: No missing values were found in the integration keys.
PASS: Country names were standardized using the approved dictionary.

Original datasets were not modified.
No integrated dataset has been created yet.


In [14]:
# Cell 14: Analyze country-year key coverage before integration

from itertools import combinations

# Build unique ISO3-Year key sets for every standardized dataset
dataset_keys = {}

for filename, df in standardized_datasets.items():

    keys = set(
        zip(
            df["ISO3"].astype(str),
            df["Year"].astype(int)
        )
    )

    dataset_keys[filename] = keys


# Calculate the union of all country-year keys
all_keys = set().union(*dataset_keys.values())

print("=" * 110)
print("PRE-INTEGRATION COUNTRY-YEAR COVERAGE AUDIT")
print("=" * 110)

print(f"Number of source datasets: {len(dataset_keys)}")
print(f"Total unique ISO3-Year keys across all datasets: {len(all_keys):,}")


# Summarize each dataset's overlap with all other datasets
coverage_records = []

for filename, keys in dataset_keys.items():

    other_keys = set().union(
        *[
            other_set
            for other_filename, other_set in dataset_keys.items()
            if other_filename != filename
        ]
    )

    overlapping_keys = keys & other_keys
    exclusive_keys = keys - other_keys

    df = standardized_datasets[filename]

    coverage_records.append({
        "Dataset": filename,
        "Rows": len(df),
        "Unique_ISO3_Year_Keys": len(keys),
        "Keys_Also_Found_In_Other_Datasets": len(overlapping_keys),
        "Keys_Unique_To_This_Dataset": len(exclusive_keys),
        "Pct_Keys_Shared_With_Any_Other": round(
            100 * len(overlapping_keys) / len(keys), 2
        ) if keys else 0,
        "Min_Year": int(df["Year"].min()),
        "Max_Year": int(df["Year"].max()),
        "Unique_ISO3": df["ISO3"].nunique()
    })


coverage_summary = pd.DataFrame(coverage_records)

print("\n" + "=" * 110)
print("DATASET-LEVEL OVERLAP SUMMARY")
print("=" * 110)

display(coverage_summary)


# Build a pairwise overlap matrix
dataset_names = list(dataset_keys.keys())

overlap_matrix = pd.DataFrame(
    index=dataset_names,
    columns=dataset_names,
    dtype="int64"
)

for name_a in dataset_names:
    for name_b in dataset_names:
        overlap_matrix.loc[name_a, name_b] = len(
            dataset_keys[name_a] & dataset_keys[name_b]
        )

overlap_matrix.index.name = "Dataset"

print("\n" + "=" * 110)
print("PAIRWISE ISO3-YEAR OVERLAP MATRIX")
print("=" * 110)

display(overlap_matrix)

print("\nInterpretation:")
print("- Diagonal values represent the number of unique keys in each dataset.")
print("- Off-diagonal values represent the number of exact ISO3-Year keys shared by two datasets.")
print("- Full outer integration should preserve the full union of unique keys.")

print("\nNo datasets were merged or modified in this cell.")

PRE-INTEGRATION COUNTRY-YEAR COVERAGE AUDIT
Number of source datasets: 7
Total unique ISO3-Year keys across all datasets: 12,855

DATASET-LEVEL OVERLAP SUMMARY


,Dataset,Rows,Unique_ISO3_Year_Keys,Keys_Also_Found_In_Other_Datasets,Keys_Unique_To_This_Dataset,Pct_Keys_Shared_With_Any_Other,Min_Year,Max_Year,Unique_ISO3
0,demographics_clean.csv,12804,12804,9264,3540,72.35,1960,2025,194
1,socioeconomic_education_clean.csv,8634,8634,8634,0,100.00,1970,2025,194
2,healthcare_workforce_clean.csv,4624,4624,4621,3,99.94,1971,2024,194
3,healthcare_capacity_clean.csv,2958,2958,2958,0,100.00,2000,2023,193
4,healthcare_coverage_clean.csv,4680,4680,4680,0,100.00,2000,2023,195
5,health_financing_clean.csv,4612,4612,4612,0,100.00,2000,2024,195
6,financial_access_equity_clean.csv,1079,1079,1079,0,100.00,1985,2024,165



PAIRWISE ISO3-YEAR OVERLAP MATRIX


,demographics_clean.csv,socioeconomic_education_clean.csv,healthcare_workforce_clean.csv,healthcare_capacity_clean.csv,healthcare_coverage_clean.csv,health_financing_clean.csv,financial_access_equity_clean.csv
Dataset,,,,,,,
demographics_clean.csv,12804.0,8634.0,4605.0,2945.0,4632.0,4564.0,1077.0
socioeconomic_education_clean.csv,8634.0,8634.0,4044.0,2918.0,4532.0,4497.0,1070.0
healthcare_workforce_clean.csv,4605.0,4044.0,4624.0,2469.0,3229.0,3209.0,869.0
healthcare_capacity_clean.csv,2945.0,2918.0,2469.0,2958.0,2958.0,2932.0,802.0
healthcare_coverage_clean.csv,4632.0,4532.0,3229.0,2958.0,4680.0,4579.0,981.0
health_financing_clean.csv,4564.0,4497.0,3209.0,2932.0,4579.0,4612.0,973.0
financial_access_equity_clean.csv,1077.0,1070.0,869.0,802.0,981.0,973.0,1079.0



Interpretation:
- Diagonal values represent the number of unique keys in each dataset.
- Off-diagonal values represent the number of exact ISO3-Year keys shared by two datasets.
- Full outer integration should preserve the full union of unique keys.

No datasets were merged or modified in this cell.


In [15]:
# Cell 15: Build and validate an integration preview
# This cell does NOT save or overwrite any source files.

from functools import reduce

KEYS = ["ISO3", "Year"]

print("=" * 100)
print("INTEGRATION PREVIEW")
print("=" * 100)

# --------------------------------------------------
# 1. Build the complete set of country-year keys
# --------------------------------------------------

all_key_frames = []

for filename, df in standardized_datasets.items():

    # Keep only the canonical key columns
    keys_df = df[KEYS].copy()

    all_key_frames.append(keys_df)

all_country_year_keys = (
    pd.concat(all_key_frames, ignore_index=True)
    .drop_duplicates(subset=KEYS)
    .sort_values(KEYS)
    .reset_index(drop=True)
)

print(
    "Unique country-year keys across all datasets:",
    len(all_country_year_keys)
)

# --------------------------------------------------
# 2. Assign the canonical country name
# --------------------------------------------------

integrated_preview = all_country_year_keys.copy()

integrated_preview["Country"] = (
    integrated_preview["ISO3"].map(canonical_names)
)

if integrated_preview["Country"].isna().any():
    raise ValueError(
        "Some integration keys could not be mapped to canonical country names."
    )

# Arrange the initial columns
integrated_preview = integrated_preview[
    ["ISO3", "Country", "Year"]
]

# --------------------------------------------------
# 3. Add indicator columns from each dataset
# --------------------------------------------------

merge_log = []

for filename, df in standardized_datasets.items():

    # Country is already standardized in the integration preview.
    # Keep the country-year key and all indicator columns.
    payload = df.drop(columns=["Country"]).copy()

    rows_before = len(integrated_preview)

    integrated_preview = integrated_preview.merge(
        payload,
        on=KEYS,
        how="left",
        validate="one_to_one"
    )

    rows_after = len(integrated_preview)

    merge_log.append({
        "Dataset": filename,
        "Rows_Before": rows_before,
        "Rows_After": rows_after,
        "Row_Count_Changed": rows_before != rows_after
    })

    if rows_after != rows_before:
        raise ValueError(
            f"Unexpected row-count change while merging {filename}."
        )

    print(
        f"[PASS] {filename}: "
        f"{rows_before:,} -> {rows_after:,} rows"
    )

# --------------------------------------------------
# 4. Validate the final preview structure
# --------------------------------------------------

duplicate_keys = int(
    integrated_preview.duplicated(subset=KEYS).sum()
)

missing_key_values = int(
    integrated_preview[KEYS].isna().sum().sum()
)

expected_keys = len(all_country_year_keys)
actual_keys = len(integrated_preview)

print("\n" + "=" * 100)
print("FINAL INTEGRATION PREVIEW CHECKS")
print("=" * 100)

print(f"Expected unique country-year keys: {expected_keys:,}")
print(f"Actual rows in integration preview: {actual_keys:,}")
print(f"Duplicate ISO3 + Year keys: {duplicate_keys}")
print(f"Missing values in ISO3 or Year: {missing_key_values}")
print(f"Number of columns: {len(integrated_preview.columns)}")
print(f"Number of unique countries: {integrated_preview['ISO3'].nunique()}")
print(
    f"Year range: "
    f"{integrated_preview['Year'].min()}–"
    f"{integrated_preview['Year'].max()}"
)

print("\n" + "=" * 100)
print("MERGE ROW-COUNT AUDIT")
print("=" * 100)

display(pd.DataFrame(merge_log))

# --------------------------------------------------
# 5. Report missing values in the integrated preview
# --------------------------------------------------

print("\n" + "=" * 100)
print("MISSING VALUES BY COLUMN IN THE INTEGRATION PREVIEW")
print("=" * 100)

missing_preview = (
    integrated_preview
    .isna()
    .sum()
    .rename("Missing_Count")
    .to_frame()
)

missing_preview["Missing_Pct"] = (
    missing_preview["Missing_Count"]
    .div(len(integrated_preview))
    .mul(100)
    .round(2)
)

display(
    missing_preview.sort_values(
        "Missing_Count",
        ascending=False
    )
)

# --------------------------------------------------
# 6. Final preview status
# --------------------------------------------------

if (
    actual_keys == expected_keys
    and duplicate_keys == 0
    and missing_key_values == 0
    and not pd.DataFrame(merge_log)["Row_Count_Changed"].any()
):
    print("\nSTATUS: PASS — Integration preview passed structural checks.")
else:
    print("\nSTATUS: REVIEW REQUIRED — One or more checks need investigation.")

print("\nNo integrated CSV was saved.")
print("Original source files were not modified.")

INTEGRATION PREVIEW
Unique country-year keys across all datasets: 12855
[PASS] demographics_clean.csv: 12,855 -> 12,855 rows
[PASS] socioeconomic_education_clean.csv: 12,855 -> 12,855 rows
[PASS] healthcare_workforce_clean.csv: 12,855 -> 12,855 rows
[PASS] healthcare_capacity_clean.csv: 12,855 -> 12,855 rows
[PASS] healthcare_coverage_clean.csv: 12,855 -> 12,855 rows
[PASS] health_financing_clean.csv: 12,855 -> 12,855 rows
[PASS] financial_access_equity_clean.csv: 12,855 -> 12,855 rows

FINAL INTEGRATION PREVIEW CHECKS
Expected unique country-year keys: 12,855
Actual rows in integration preview: 12,855
Duplicate ISO3 + Year keys: 0
Missing values in ISO3 or Year: 0
Number of columns: 27
Number of unique countries: 196
Year range: 1960–2025

MERGE ROW-COUNT AUDIT


,Dataset,Rows_Before,Rows_After,Row_Count_Changed
0,demographics_clean.csv,12855,12855,False
1,socioeconomic_education_clean.csv,12855,12855,False
2,healthcare_workforce_clean.csv,12855,12855,False
3,healthcare_capacity_clean.csv,12855,12855,False
4,healthcare_coverage_clean.csv,12855,12855,False
5,health_financing_clean.csv,12855,12855,False
6,financial_access_equity_clean.csv,12855,12855,False



MISSING VALUES BY COLUMN IN THE INTEGRATION PREVIEW


,Missing_Count,Missing_Pct
Equity_Gap_Pct_Points,11817,91.93
Q5_Richest_Pct,11817,91.93
Q4_Pct,11817,91.93
Q3_Pct,11817,91.93
Q2_Pct,11817,91.93
Q1_Poorest_Pct,11817,91.93
Impoverishing_Spending_Pct,11799,91.79
Large_Spending_Pct,11799,91.79
Financial_Hardship_Total_Pct,11776,91.61
Adult_Literacy_Rate,11753,91.43



STATUS: PASS — Integration preview passed structural checks.

No integrated CSV was saved.
Original source files were not modified.


In [16]:
# Cell 16: Verify source values were preserved after integration
# This cell performs validation only. It does not save or modify any files.

KEYS = ["ISO3", "Year"]

integration_keys = set(
    zip(
        integrated_preview["ISO3"].astype(str),
        integrated_preview["Year"].astype(int)
    )
)

key_preservation_results = []
value_preservation_results = []

for filename, source_df in standardized_datasets.items():

    source_keys = set(
        zip(
            source_df["ISO3"].astype(str),
            source_df["Year"].astype(int)
        )
    )

    missing_from_integration = source_keys - integration_keys

    key_preservation_results.append({
        "Dataset": filename,
        "Source_Keys": len(source_keys),
        "Keys_Found_In_Integration": len(source_keys & integration_keys),
        "Keys_Missing_From_Integration": len(missing_from_integration)
    })

    # Compare the non-null count of each source indicator
    # against its corresponding column in the integrated preview.
    indicator_columns = [
        col for col in source_df.columns
        if col not in ["ISO3", "Country", "Year"]
    ]

    for column in indicator_columns:

        source_non_null = int(source_df[column].notna().sum())
        integrated_non_null = int(
            integrated_preview[column].notna().sum()
        )

        value_preservation_results.append({
            "Dataset": filename,
            "Indicator": column,
            "Source_NonNull_Count": source_non_null,
            "Integrated_NonNull_Count": integrated_non_null,
            "Difference": integrated_non_null - source_non_null,
            "Status": (
                "PASS"
                if source_non_null == integrated_non_null
                else "REVIEW"
            )
        })


key_preservation_summary = pd.DataFrame(key_preservation_results)
value_preservation_summary = pd.DataFrame(value_preservation_results)

print("=" * 100)
print("SOURCE KEY PRESERVATION")
print("=" * 100)

display(key_preservation_summary)

print("\n" + "=" * 100)
print("INDICATOR VALUE PRESERVATION")
print("=" * 100)

display(value_preservation_summary)

# Overall checks
keys_ok = (
    key_preservation_summary["Keys_Missing_From_Integration"].eq(0).all()
)

values_ok = (
    value_preservation_summary["Difference"].eq(0).all()
)

print("\n" + "=" * 100)

if keys_ok and values_ok:
    print("PASS: All source country-year keys were preserved.")
    print("PASS: All non-missing indicator counts were preserved.")
else:
    print("REVIEW REQUIRED: One or more preservation checks failed.")

print("\nNo source data was changed.")
print("No output file was saved.")

SOURCE KEY PRESERVATION


,Dataset,Source_Keys,Keys_Found_In_Integration,Keys_Missing_From_Integration
0,demographics_clean.csv,12804,12804,0
1,socioeconomic_education_clean.csv,8634,8634,0
2,healthcare_workforce_clean.csv,4624,4624,0
3,healthcare_capacity_clean.csv,2958,2958,0
4,healthcare_coverage_clean.csv,4680,4680,0
5,health_financing_clean.csv,4612,4612,0
6,financial_access_equity_clean.csv,1079,1079,0



INDICATOR VALUE PRESERVATION


,Dataset,Indicator,Source_NonNull_Count,Integrated_NonNull_Count,Difference,Status
0,demographics_clean.csv,Population,12774,12774,0,PASS
1,demographics_clean.csv,Age_0_14_Pct,12804,12804,0,PASS
2,demographics_clean.csv,Age_65_Plus_Pct,12804,12804,0,PASS
3,demographics_clean.csv,Urban_Population_Pct,12804,12804,0,PASS
4,socioeconomic_education_clean.csv,GDP_Per_Capita_PPP_2021,6630,6630,0,PASS
5,socioeconomic_education_clean.csv,Secondary_Enrollment_Gross_Pct,6023,6023,0,PASS
6,socioeconomic_education_clean.csv,Adult_Literacy_Rate,1102,1102,0,PASS
7,healthcare_workforce_clean.csv,Doctors_Total,4293,4293,0,PASS
8,healthcare_workforce_clean.csv,Doctors_Per_10k,4280,4280,0,PASS
9,healthcare_workforce_clean.csv,Nurses_Midwives_Per_10k,3374,3374,0,PASS



PASS: All source country-year keys were preserved.
PASS: All non-missing indicator counts were preserved.

No source data was changed.
No output file was saved.


In [17]:
# Cell 17: Indicator Coverage Analysis

total_rows = len(integrated_preview)

key_columns = ["ISO3", "Country", "Year"]

indicator_columns = [
    col for col in integrated_preview.columns
    if col not in key_columns
]

coverage_records = []

for column in indicator_columns:

    available_count = int(integrated_preview[column].notna().sum())
    missing_count = int(integrated_preview[column].isna().sum())

    coverage_records.append({
        "Indicator": column,
        "Available_Values": available_count,
        "Missing_Values": missing_count,
        "Coverage_Pct": round(
            100 * available_count / total_rows, 2
        ),
        "Missing_Pct": round(
            100 * missing_count / total_rows, 2
        )
    })


indicator_coverage = (
    pd.DataFrame(coverage_records)
    .sort_values("Coverage_Pct", ascending=False)
    .reset_index(drop=True)
)

print("=" * 100)
print("INTEGRATED DATASET — INDICATOR COVERAGE")
print("=" * 100)

print(f"Total integrated rows: {total_rows:,}")
print(f"Number of indicators: {len(indicator_columns)}")

display(indicator_coverage)

print("\n" + "=" * 100)
print("COVERAGE GROUPS")
print("=" * 100)

coverage_groups = pd.cut(
    indicator_coverage["Coverage_Pct"],
    bins=[-0.01, 25, 50, 75, 100],
    labels=[
        "Low: 0–25%",
        "Moderate: >25–50%",
        "High: >50–75%",
        "Very High: >75–100%"
    ]
)

display(
    indicator_coverage.assign(
        Coverage_Group=coverage_groups
    )[["Indicator", "Coverage_Pct", "Coverage_Group"]]
)

print("\nNo source data was modified.")
print("No final integrated file was saved.")

INTEGRATED DATASET — INDICATOR COVERAGE
Total integrated rows: 12,855
Number of indicators: 24


,Indicator,Available_Values,Missing_Values,Coverage_Pct,Missing_Pct
0,Age_65_Plus_Pct,12804,51,99.60,0.40
1,Urban_Population_Pct,12804,51,99.60,0.40
2,Age_0_14_Pct,12804,51,99.60,0.40
3,Population,12774,81,99.37,0.63
4,GDP_Per_Capita_PPP_2021,6630,6225,51.58,48.42
5,Secondary_Enrollment_Gross_Pct,6023,6832,46.85,53.15
6,UHC_Service_Coverage_Index,4680,8175,36.41,63.59
7,OOP_Pct_Current_Health_Exp,4610,8245,35.86,64.14
8,Health_Exp_Pct_GDP,4610,8245,35.86,64.14
9,Health_Exp_Per_Capita_PPP,4609,8246,35.85,64.15



COVERAGE GROUPS


,Indicator,Coverage_Pct,Coverage_Group
0,Age_65_Plus_Pct,99.60,Very High: >75–100%
1,Urban_Population_Pct,99.60,Very High: >75–100%
2,Age_0_14_Pct,99.60,Very High: >75–100%
3,Population,99.37,Very High: >75–100%
4,GDP_Per_Capita_PPP_2021,51.58,High: >50–75%
5,Secondary_Enrollment_Gross_Pct,46.85,Moderate: >25–50%
6,UHC_Service_Coverage_Index,36.41,Moderate: >25–50%
7,OOP_Pct_Current_Health_Exp,35.86,Moderate: >25–50%
8,Health_Exp_Pct_GDP,35.86,Moderate: >25–50%
9,Health_Exp_Per_Capita_PPP,35.85,Moderate: >25–50%



No source data was modified.
No final integrated file was saved.


In [19]:
# Cell 18: Pairwise Indicator Availability for Planned Analyses

total_rows = len(integrated_preview)

# Planned analytical comparisons
analysis_pairs = [
    (
        "UHC vs GDP per Capita PPP",
        "UHC_Service_Coverage_Index",
        "GDP_Per_Capita_PPP_2021"
    ),
    (
        "UHC vs Secondary Enrollment",
        "UHC_Service_Coverage_Index",
        "Secondary_Enrollment_Gross_Pct"
    ),
    (
        "UHC vs Adult Literacy",
        "UHC_Service_Coverage_Index",
        "Adult_Literacy_Rate"
    ),
    (
        "UHC vs Doctors per 10k",
        "UHC_Service_Coverage_Index",
        "Doctors_Per_10k"
    ),
    (
        "UHC vs Nurses and Midwives per 10k",
        "UHC_Service_Coverage_Index",
        "Nurses_Midwives_Per_10k"
    ),
    (
        "UHC vs Hospital Beds per 10k",
        "UHC_Service_Coverage_Index",
        "Hospital_Beds_Per_10k"
    ),
    (
        "UHC vs Health Expenditure per Capita PPP",
        "UHC_Service_Coverage_Index",
        "Health_Exp_Per_Capita_PPP"
    ),
    (
        "UHC vs Health Expenditure % GDP",
        "UHC_Service_Coverage_Index",
        "Health_Exp_Pct_GDP"
    ),
    (
        "UHC vs Out-of-Pocket Expenditure",
        "UHC_Service_Coverage_Index",
        "OOP_Pct_Current_Health_Exp"
    ),
    (
        "Financial Hardship vs UHC",
        "Financial_Hardship_Total_Pct",
        "UHC_Service_Coverage_Index"
    ),
    (
        "Financial Hardship vs Out-of-Pocket Expenditure",
        "Financial_Hardship_Total_Pct",
        "OOP_Pct_Current_Health_Exp"
    ),
    (
        "Financial Hardship vs GDP per Capita PPP",
        "Financial_Hardship_Total_Pct",
        "GDP_Per_Capita_PPP_2021"
    ),
]

pairwise_results = []

for comparison, indicator_a, indicator_b in analysis_pairs:

    # Ensure both indicators exist in the integrated preview
    if (
        indicator_a not in integrated_preview.columns
        or indicator_b not in integrated_preview.columns
    ):
        pairwise_results.append({
            "Comparison": comparison,
            "Indicator_A": indicator_a,
            "Indicator_B": indicator_b,
            "Shared_Observations": None,
            "Unique_Countries": None,
            "First_Year": None,
            "Last_Year": None,
            "Coverage_Pct_Of_Integrated_Rows": None,
            "Status": "MISSING COLUMN"
        })
        continue

    # Count rows where both indicators are available
    valid_mask = (
        integrated_preview[indicator_a].notna()
        & integrated_preview[indicator_b].notna()
    )

    common_data = integrated_preview.loc[
        valid_mask,
        ["ISO3", "Year", indicator_a, indicator_b]
    ]

    n_common = len(common_data)

    pairwise_results.append({
        "Comparison": comparison,
        "Indicator_A": indicator_a,
        "Indicator_B": indicator_b,
        "Shared_Observations": n_common,
        "Unique_Countries": common_data["ISO3"].nunique(),
        "First_Year": (
            int(common_data["Year"].min())
            if n_common else None
        ),
        "Last_Year": (
            int(common_data["Year"].max())
            if n_common else None
        ),
        "Coverage_Pct_Of_Integrated_Rows": round(
            100 * n_common / total_rows, 2
        ),
        "Status": "REVIEW" if n_common == 0 else "CHECKED"
    })


pairwise_coverage = pd.DataFrame(pairwise_results)

print("=" * 110)
print("PAIRWISE DATA AVAILABILITY FOR PLANNED ANALYSES")
print("=" * 110)

display(
    pairwise_coverage.sort_values(
        "Shared_Observations",
        ascending=False
    ).reset_index(drop=True)
)

print("\n" + "=" * 110)
print("IMPORTANT INTERPRETATION")
print("=" * 110)

print(
    "These results measure data availability only. "
    "They do not establish statistical relationships or causation."
)

print(
    "Financial hardship observations are based on survey years; "
    "they should not be treated as a complete annual series."
)

print("\nNo source datasets were modified.")
print("No final integrated dataset was saved.")

PAIRWISE DATA AVAILABILITY FOR PLANNED ANALYSES


,Comparison,Indicator_A,Indicator_B,Shared_Observations,Unique_Countries,First_Year,Last_Year,Coverage_Pct_Of_Integrated_Rows,Status
0,UHC vs Health Expenditure % GDP,UHC_Service_Coverage_Index,Health_Exp_Pct_GDP,4577,194,2000,2023,35.60,CHECKED
1,UHC vs Out-of-Pocket Expenditure,UHC_Service_Coverage_Index,OOP_Pct_Current_Health_Exp,4577,194,2000,2023,35.60,CHECKED
2,UHC vs Health Expenditure per Capita PPP,UHC_Service_Coverage_Index,Health_Exp_Per_Capita_PPP,4576,194,2000,2023,35.60,CHECKED
3,UHC vs GDP per Capita PPP,UHC_Service_Coverage_Index,GDP_Per_Capita_PPP_2021,4450,186,2000,2023,34.62,CHECKED
4,UHC vs Secondary Enrollment,UHC_Service_Coverage_Index,Secondary_Enrollment_Gross_Pct,3304,190,2000,2023,25.70,CHECKED
5,UHC vs Hospital Beds per 10k,UHC_Service_Coverage_Index,Hospital_Beds_Per_10k,2958,193,2000,2023,23.01,CHECKED
6,UHC vs Doctors per 10k,UHC_Service_Coverage_Index,Doctors_Per_10k,2956,192,2000,2023,22.99,CHECKED
7,UHC vs Nurses and Midwives per 10k,UHC_Service_Coverage_Index,Nurses_Midwives_Per_10k,2839,194,2000,2023,22.08,CHECKED
8,Financial Hardship vs GDP per Capita PPP,Financial_Hardship_Total_Pct,GDP_Per_Capita_PPP_2021,1066,161,1990,2024,8.29,CHECKED
9,Financial Hardship vs UHC,Financial_Hardship_Total_Pct,UHC_Service_Coverage_Index,981,159,2000,2023,7.63,CHECKED



IMPORTANT INTERPRETATION
These results measure data availability only. They do not establish statistical relationships or causation.
Financial hardship observations are based on survey years; they should not be treated as a complete annual series.

No source datasets were modified.
No final integrated dataset was saved.


In [20]:
# Cell 19: Exact value preservation audit
# Compare every source indicator with its corresponding integrated value.

KEYS = ["ISO3", "Year"]

integrated_indexed = (
    integrated_preview
    .set_index(KEYS)
    .sort_index()
)

value_check_results = []

for filename, source_df in standardized_datasets.items():

    source_indexed = (
        source_df
        .set_index(KEYS)
        .sort_index()
    )

    # Retrieve integrated rows using the exact source keys
    integrated_source_rows = integrated_indexed.reindex(
        source_indexed.index
    )

    indicator_columns = [
        col for col in source_df.columns
        if col not in ["ISO3", "Country", "Year"]
    ]

    for column in indicator_columns:

        source_values = source_indexed[column]
        integrated_values = integrated_source_rows[column]

        # Compare numeric values while treating NaN == NaN
        source_numeric = pd.to_numeric(
            source_values, errors="coerce"
        ).to_numpy(dtype="float64", na_value=np.nan)

        integrated_numeric = pd.to_numeric(
            integrated_values, errors="coerce"
        ).to_numpy(dtype="float64", na_value=np.nan)

        equal_mask = np.isclose(
            source_numeric,
            integrated_numeric,
            rtol=0,
            atol=0,
            equal_nan=True
        )

        mismatch_count = int((~equal_mask).sum())

        value_check_results.append({
            "Dataset": filename,
            "Indicator": column,
            "Compared_Records": len(source_values),
            "Mismatch_Count": mismatch_count,
            "Status": "PASS" if mismatch_count == 0 else "REVIEW"
        })


exact_value_audit = pd.DataFrame(value_check_results)

print("=" * 100)
print("EXACT SOURCE-TO-INTEGRATED VALUE PRESERVATION")
print("=" * 100)

display(exact_value_audit)

print("\n" + "=" * 100)
print("AUDIT RESULT")
print("=" * 100)

if exact_value_audit["Mismatch_Count"].eq(0).all():
    print("PASS: Every checked indicator value matches its source value.")
    print("PASS: No mismatches were detected in the integration preview.")
else:
    print("REVIEW REQUIRED: Some values differ from their source values.")
    display(
        exact_value_audit.loc[
            exact_value_audit["Mismatch_Count"] > 0
        ]
    )

print("\nNo source files were modified.")
print("No final integrated file was saved.")

EXACT SOURCE-TO-INTEGRATED VALUE PRESERVATION


,Dataset,Indicator,Compared_Records,Mismatch_Count,Status
0,demographics_clean.csv,Population,12804,0,PASS
1,demographics_clean.csv,Age_0_14_Pct,12804,0,PASS
2,demographics_clean.csv,Age_65_Plus_Pct,12804,0,PASS
3,demographics_clean.csv,Urban_Population_Pct,12804,0,PASS
4,socioeconomic_education_clean.csv,GDP_Per_Capita_PPP_2021,8634,0,PASS
5,socioeconomic_education_clean.csv,Secondary_Enrollment_Gross_Pct,8634,0,PASS
6,socioeconomic_education_clean.csv,Adult_Literacy_Rate,8634,0,PASS
7,healthcare_workforce_clean.csv,Doctors_Total,4624,0,PASS
8,healthcare_workforce_clean.csv,Doctors_Per_10k,4624,0,PASS
9,healthcare_workforce_clean.csv,Nurses_Midwives_Per_10k,4624,0,PASS



AUDIT RESULT
PASS: Every checked indicator value matches its source value.
PASS: No mismatches were detected in the integration preview.

No source files were modified.
No final integrated file was saved.


In [21]:
# Cell 20: Save final integration deliverables
# Original team datasets remain unchanged.

from pathlib import Path
import pandas as pd

# --------------------------------------------------
# 1. Define output directory
# --------------------------------------------------

OUTPUT_DIR = DATA_DIR.parent / "03_Integrated_Datasets"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 2. Final validation gates
# --------------------------------------------------

KEYS = ["ISO3", "Year"]

assert integrated_preview.shape == (12855, 27), (
    f"Unexpected integration shape: {integrated_preview.shape}"
)

assert integrated_preview.duplicated(subset=KEYS).sum() == 0, (
    "Duplicate ISO3 + Year keys detected."
)

assert integrated_preview[KEYS].isna().sum().sum() == 0, (
    "Missing values found in integration keys."
)

assert integrated_preview["Country"].notna().all(), (
    "Missing canonical country names found."
)

assert key_preservation_summary[
    "Keys_Missing_From_Integration"
].eq(0).all(), (
    "Some source keys are missing from the integration."
)

assert exact_value_audit["Mismatch_Count"].eq(0).all(), (
    "Some source indicator values differ from the integration."
)

assert value_preservation_summary["Difference"].eq(0).all(), (
    "Source indicator non-null counts were not preserved."
)

# --------------------------------------------------
# 3. Prepare validation summary
# --------------------------------------------------

validation_records = [
    {
        "Check": "Expected integrated rows",
        "Result": len(integrated_preview),
        "Status": "PASS"
    },
    {
        "Check": "Integrated columns",
        "Result": len(integrated_preview.columns),
        "Status": "PASS"
    },
    {
        "Check": "Unique ISO3 codes",
        "Result": integrated_preview["ISO3"].nunique(),
        "Status": "PASS"
    },
    {
        "Check": "Minimum year",
        "Result": int(integrated_preview["Year"].min()),
        "Status": "PASS"
    },
    {
        "Check": "Maximum year",
        "Result": int(integrated_preview["Year"].max()),
        "Status": "PASS"
    },
    {
        "Check": "Duplicate ISO3 + Year keys",
        "Result": int(
            integrated_preview.duplicated(subset=KEYS).sum()
        ),
        "Status": "PASS"
    },
    {
        "Check": "Missing integration key values",
        "Result": int(
            integrated_preview[KEYS].isna().sum().sum()
        ),
        "Status": "PASS"
    },
    {
        "Check": "Source keys preserved",
        "Result": int(
            key_preservation_summary[
                "Keys_Missing_From_Integration"
            ].sum()
        ),
        "Status": "PASS"
    },
    {
        "Check": "Indicator value mismatches",
        "Result": int(
            exact_value_audit["Mismatch_Count"].sum()
        ),
        "Status": "PASS"
    },
    {
        "Check": "Indicator non-null count differences",
        "Result": int(
            value_preservation_summary["Difference"].abs().sum()
        ),
        "Status": "PASS"
    },
    {
        "Check": "Financial equity records excluded from integration",
        "Result": len(equity_unmatched_review),
        "Status": "DOCUMENTED REVIEW"
    }
]

integration_validation_summary = pd.DataFrame(validation_records)

# --------------------------------------------------
# 4. Define all deliverables
# --------------------------------------------------

deliverables = {
    "integrated_healthcare_dataset.csv": integrated_preview,
    "indicator_coverage_summary.csv": indicator_coverage,
    "pairwise_analysis_coverage.csv": pairwise_coverage,
    "integration_validation_summary.csv": integration_validation_summary,
    "source_standardization_summary.csv": standardization_summary,
    "financial_equity_unmatched_review.csv": equity_unmatched_review,
}

# Protect existing files from accidental overwriting
existing_outputs = [
    filename
    for filename in deliverables
    if (OUTPUT_DIR / filename).exists()
]

if existing_outputs:
    raise FileExistsError(
        "The following output files already exist. "
        "Review them before rerunning this save step:\n"
        + "\n".join(existing_outputs)
    )

# --------------------------------------------------
# 5. Save CSV deliverables
# --------------------------------------------------

for filename, dataframe in deliverables.items():

    dataframe.to_csv(
        OUTPUT_DIR / filename,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"[SAVED] {filename} | Shape: {dataframe.shape}")

# --------------------------------------------------
# 6. Write a concise integration audit log
# --------------------------------------------------

audit_log = [
    "GLOBAL HEALTHCARE PERFORMANCE & EQUITY ANALYSIS",
    "DATA INTEGRATION AUDIT LOG",
    "",
    f"Integrated rows: {len(integrated_preview)}",
    f"Integrated columns: {len(integrated_preview.columns)}",
    f"Unique ISO3 codes: {integrated_preview['ISO3'].nunique()}",
    (
        f"Year range: {integrated_preview['Year'].min()}-"
        f"{integrated_preview['Year'].max()}"
    ),
    f"Duplicate ISO3-Year keys: {integrated_preview.duplicated(subset=KEYS).sum()}",
    f"Source value mismatches: {exact_value_audit['Mismatch_Count'].sum()}",
    (
        "Source key records missing from integration: "
        f"{key_preservation_summary['Keys_Missing_From_Integration'].sum()}"
    ),
    (
        "Financial equity records excluded due to unmatched ISO3 codes: "
        f"{len(equity_unmatched_review)}"
    ),
    "",
    "Missing values were preserved and were not imputed.",
    "Original team datasets were not modified.",
    "The integration is a country-year full-union dataset.",
    (
        "Financial hardship indicators use survey years and should not "
        "be interpreted as a complete annual series."
    ),
    "",
    "Important: Structural validation passed. The integrated dataset "
    "still requires substantive review of source definitions, units, "
    "coverage, and fitness for each planned analysis."
]

log_path = OUTPUT_DIR / "integration_audit_log.txt"

if log_path.exists():
    raise FileExistsError(
        f"{log_path.name} already exists; refusing to overwrite it."
    )

log_path.write_text(
    "\n".join(audit_log),
    encoding="utf-8"
)

print(f"[SAVED] {log_path.name}")

# --------------------------------------------------
# 7. Final confirmation
# --------------------------------------------------

print("\n" + "=" * 90)
print("INTEGRATION DELIVERABLES SAVED")
print("=" * 90)

print(f"Output folder: {OUTPUT_DIR}")

for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(f"- {path.name} ({path.stat().st_size / 1024:.2f} KB)")

print("\nAll final structural validation gates passed.")
print("Original source datasets remain unchanged.")

[SAVED] integrated_healthcare_dataset.csv | Shape: (12855, 27)
[SAVED] indicator_coverage_summary.csv | Shape: (24, 5)
[SAVED] pairwise_analysis_coverage.csv | Shape: (12, 9)
[SAVED] integration_validation_summary.csv | Shape: (11, 3)
[SAVED] source_standardization_summary.csv | Shape: (7, 11)
[SAVED] financial_equity_unmatched_review.csv | Shape: (9, 13)
[SAVED] integration_audit_log.txt

INTEGRATION DELIVERABLES SAVED
Output folder: c:\Users\talal\OneDrive\Desktop\03_Integrated_Datasets
- financial_equity_unmatched_review.csv (2.15 KB)
- indicator_coverage_summary.csv (1.04 KB)
- integrated_healthcare_dataset.csv (1822.99 KB)
- integration_audit_log.txt (0.76 KB)
- integration_validation_summary.csv (0.41 KB)
- pairwise_analysis_coverage.csv (1.50 KB)
- source_standardization_summary.csv (0.68 KB)

All final structural validation gates passed.
Original source datasets remain unchanged.
